In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os



# reading and loading CSV file

dataset_path = os.path.join(path, 'Q3_data.csv')
df_customer = pd.read_csv(dataset_path)


In [ ]:
# Task 2: Write your code here:
df_customer.head()

In [ ]:
# Task 3: Write your code here:
df_customer.info()

In [ ]:
# Task 4: Write your code here:
df_customer.describe()

In [ ]:
# Task 1: Write your code here:

# Analyzing for missing values
missing_percentage = (df_customer.isnull().sum() / len(df_customer)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
print(missing_data.head(50))
print('-------------------------------------------------------')


alot_empty_val_col = ['D_87', 'D_88', 'B_39', 'D_110', 'D_111', 'D_108', 'B_42', 'D_73', 'D_135', 'D_136', 'D_138', 'D_134', 'D_137', 'R_9', 'B_29', 'D_106']
df_clean = df_customer.drop(columns=alot_empty_val_col)

df_clean.head()


df_clean = df_clean.dropna()
print(f"After dropping missing vals: {df_clean.shape}")



In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 3: Write your code here:

categorical_cols = df_clean.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))


label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean.head(10)

In [ ]:
# Task 4: Write your code here:

scaler = StandardScaler()

df_scaled_data = pd.DataFrame(scaler.fit_transform(df_clean))

df_scaled_data


In [ ]:
# Task 5: Write your code here:

df_clean.describe()


In [ ]:
# Task 1: Write your code here:


X = df_clean[0]
y = df_clean[7]

In [ ]:
# Task 2,3,4,5: Write your code here:

def gradient_descent(X, y, num_classes, lr, n_iters=1000):
  # Get the number of samples (m) and number of features (n)
  m, n = X.shape

  # Initialize weight matrix with shape (n, num_classes)
  theta = np.zeros((n, num_classes))

  # One-hot encode the labels
  y_onehot = one_hot_encode(y, num_classes)

  losses = []

  for _ in tqdm(range(n_iters), desc="Training Multiclass Logistic Regression"):
    # Calculate the logits z
    z = np.dot(X, theta)

    # Get class probabilities using softmax
    y_pred = softmax(z)

    # Compute the gradient of Categorical Cross-Entropy with Softmax
    # ∂J/∂θ = (1/m) * X^T * (y_pred - y_onehot)
    gradient = np.dot(X.T, (y_pred - y_onehot)) / m

    # Update weights
    theta -= lr * gradient

    # Track loss
    loss = categorical_cross_entropy(y_onehot, y_pred)
    losses.append(loss)

  return theta, losses


  from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

n_splits = 3 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

sr_results = {'loss': [], 'acc': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train using gradient descent with learning rate = 0.5
  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)

  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)

  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)

  # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)

In [ ]:
# Task 1: Write your code here:
print("Evaluating models on the test set...")

# Dictionary to store performance metrics
model_performance = {}

# Iterate through each trained model
for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")

    # Make predictions on the test set (hard labels)
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # Store metrics
    model_performance[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }

    # Print metrics
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")

    # Generate and visualize Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title(f'Confusion Matrix for {model_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.show()

print("\nAll models evaluated.")


In [ ]:
# Task 2: Write your code here:
desceibe

In [ ]:
# Task Bonus: Write your code here: